In [40]:
import os
import openmc
import numpy as np
from IPython.display import Image, display
import matplotlib.pyplot as plt
from datetime import datetime


In [41]:
# pulizia per sicurezza

os.system("rm *.xml")
os.system("rm *.out")
os.system("rm *.png")

0

In [42]:
param_file = 'parametri.txt'
moltiplicatore = 1.0
water = 0.15
is_first_run = True

VARIABILI IMPORTANTI

In [43]:
def load_parameters(filename='parametri.txt'):
    """
    Legge i parametri dal file txt e li restituisce come un dizionario.
    """
    params = {}
    # Forniamo np al contesto di esecuzione per gestire np.sqrt()
    context = {'np': np} 
    
    with open(filename, 'r') as f:
        code = f.read()
        exec(code, context, params)
    
    # Rimuoviamo 'np' e altri built-in dal dizionario finale
    return {k: v for k, v in params.items() if not k.startswith('__') and k != 'np'}

p = load_parameters(param_file)

# --- Parametri Geometrici ---
R_CORE    = p['R_CORE'] * moltiplicatore
REF_SIDE  = p['REF_SIDE']
REF_BOT   = p['REF_BOT']
H_CORE    = p['H_CORE']

# --- Parametri Logici ---
R_PIPE    = p['R_PIPE']

# --- Parametri Pin/Lattice ---
R_FUEL    = p['R_FUEL']
CLAD_THICK= p['CLAD_THICK']
R_WATER_THICK = p['R_WATER_THICK']
R_PIN     = p['R_PIN']
PITCH     = p['PITCH'] * moltiplicatore

# --- Parametri Target ---
R_TARGET  = p['R_TARGET']
H_TARGET  = p['H_TARGET']
P_TARGET  = p['P_TARGET']

# --- Sorgente e Simulazione ---
S             = p['S']

bin_ris       = p['bin_ris']

batches       = p['batches_auto']
particles     = p['particles_auto']
inactive      = p['inactive_auto']

arricchimento = p['arricchimento']
mix = water
pressione_funzionamento = p['Pressione_funzionamento']

T_water_HZP = p['T_water_HZP']
T_water_HFP = p['T_water_HFP']
T_fuel_HFP = p['T_fuel_HFP']
T_fuel_HZP = p['T_fuel_HZP']

MATERIALI

In [44]:
fuel = openmc.Material(name='fuel')
fuel.add_nuclide('U235', arricchimento)
fuel.add_nuclide('U238', 1.0 - arricchimento)
fuel.add_nuclide('O16', 2.0)
fuel.set_density('g/cm3', 10.96)

# materiale di rivestimento
cladding = openmc.Material (name = 'cladding')
cladding.add_element('Zr', 1.0) 
cladding.set_density('g/cm3', 6.49)

# materiale di moderatore
water = openmc.Material (name = 'water')
water.add_nuclide('H1', mix)
water.add_nuclide('H2', 1-mix)
water.add_element('O', 1.0)
water.set_density('g/cm3', 1.10)
water.add_s_alpha_beta('c_D_in_D2O') # aggiungo la tabella di scattering termico
water.add_s_alpha_beta('c_H_in_H2O') # aggiungo la tabella di scattering termico
# n.b.  le sezioni d'urto veloci sono già incluse di default


# aggiungo un riflettore 
reflector = openmc.Material(name='reflector')
reflector.add_element('C',1.0)
reflector.set_density('g/cm3', 1.75) 

# voglio una sorgente puntiforme di neutroni in un punto specifico
marker_mat = openmc.Material(name='marker')
marker_mat.set_density('g/cm3', 1e-10) # Densità quasi nulla per non disturbare la fisica
marker_mat.add_nuclide('He4', 1.0) # elio, non interagisce praticamente

void_air = openmc.Material(name='void_air')
void_air.set_density('g/cm3', 1e-10) # Densità
void_air.add_nuclide('N14', 1 )

mat_target = openmc.Material(name='mat_target')
mat_target.add_nuclide('U235', arricchimento)
mat_target.add_nuclide('U238', 1.0 - arricchimento)
mat_target.add_nuclide('O16', 2.0)
mat_target.set_density('g/cm3', 10.96)

materials = openmc.Materials([fuel, cladding, water, reflector, marker_mat, void_air, mat_target])
materials.cross_sections = "/home/bossi_ricky/openmc_data/mcnp_endfb71/cross_sections.xml"
materials.export_to_xml()

GEOMETRIA

In [45]:
# --- 0. PIANI Z (ASSOLUTI) ---
z_bot_ref = openmc.ZPlane(z0=-REF_BOT, boundary_type='vacuum')
z_bot_core = openmc.ZPlane(z0=0.0)
z_top_core = openmc.ZPlane(z0=H_CORE)
z_top_world = openmc.ZPlane(z0=H_CORE + 5.0, boundary_type='vacuum')

# Piani Z Target
z_mid = H_CORE - P_TARGET
z_tgt_bot = openmc.ZPlane(z0=z_mid - (H_TARGET / 2.0))
z_tgt_top = openmc.ZPlane(z0=z_mid + (H_TARGET / 2.0))

# Superficie Sorgente
sphere_src = openmc.Sphere(x0=0, y0=0, z0=H_CORE + 2, r=0.5)

# --- 1. CALCOLO DIMENSIONI TAGLIO CENTRALE ---
# Calcoliamo quanto deve essere grande il buco esagonale centrale
# Rimuoviamo gli anelli che cadono dentro R_PIPE
n_rings_removed = int(np.ceil(R_PIPE / PITCH))
# Il lato del prisma che useremo per "bucare" il lattice:
EDGE_PIPE_LARGE = (n_rings_removed - 0.5) * PITCH
if EDGE_PIPE_LARGE == 0: EDGE_PIPE_LARGE = PITCH # Sicurezza minima

# --- 2. FORME GEOMETRICHE BASE ---

EDGE_FUEL = R_FUEL
EDGE_CLAD = R_FUEL + CLAD_THICK

# A. Prismi PIN
hex_fuel_prism = openmc.model.get_hexagonal_prism(edge_length=EDGE_FUEL, orientation='x')
hex_clad_prism = openmc.model.get_hexagonal_prism(edge_length=EDGE_CLAD, orientation='x')

# B. Prisma "TAGLIO" CENTRALE (Il buco grande nel reticolo)
hex_pipe_prism = openmc.model.get_hexagonal_prism(edge_length=EDGE_PIPE_LARGE, orientation='x')

# C. Cilindro Target (Unico centrale)
cyl_target = openmc.ZCylinder(r=R_TARGET)


# --- 3. UNIVERSI ---

# A. Universo PIN (Fuel -> Clad -> Water)
c_fuel = openmc.Cell(fill=fuel, region=hex_fuel_prism)
c_clad = openmc.Cell(fill=cladding, region=hex_clad_prism & ~hex_fuel_prism)
# Acqua definita "ovunque fuori dal cladding". Il confine sarà dato dal lattice.
c_water_pin = openmc.Cell(fill=water, region=~hex_clad_prism)

u_pin = openmc.Universe(cells=[c_fuel, c_clad, c_water_pin])


# C. Universo ACQUA ESTERNA (Riflettore/Gap)
c_water_full = openmc.Cell(fill=water)
u_water = openmc.Universe(cells=[c_water_full])


# --- 4. LATTICE (SOLO COMBUSTIBILE) ---
# Creiamo un lattice uniforme pieno SOLO di pin.
# Il buco centrale lo creiamo geometricamente nello step successivo, non qui.

num_rings = int(np.ceil(R_CORE / PITCH)) + 1
hex_core_edge = num_rings * PITCH
hex_ref_edge = hex_core_edge + REF_SIDE

lattice_universes = []
for i in range(num_rings):
    if i == 0:
        lattice_universes.append([u_pin])
    else:
        lattice_universes.append([u_pin] * (6 * i))

lattice = openmc.HexLattice(name='core_hex_lattice')
lattice.center = (0.0, 0.0)
lattice.pitch = (PITCH,)
lattice.outer = u_water
lattice.universes = lattice_universes[::-1] # Inversione fondamentale: Esterno -> Interno
lattice.orientation = 'x'

# --- 5. ASSEMBLAGGIO FINALE (GEOMETRIA DIRETTA) ---

prism_core = openmc.model.get_hexagonal_prism(edge_length=hex_core_edge, orientation='x')
prism_reflector = openmc.model.get_hexagonal_prism(edge_length=hex_ref_edge, orientation='x', boundary_type='vacuum')

# A. Cella Core (Lattice CON BUCO)
# ---------------------------------------------------------
# Prendiamo il prisma grande del core, e sottraiamo (~) il prisma del canale centrale.
# In questo modo il lattice viene "tagliato" via dal centro.
region_core = prism_core & ~hex_pipe_prism & +z_bot_core & -z_top_core
c_main_core = openmc.Cell(fill=lattice, region=region_core)


# B. Canale Centrale (Target + Vuoto) - DEFINIZIONE DIRETTA
# ---------------------------------------------------------
# 1. Definiamo il volume TOTALE del canale esagonale (da 0 a H_CORE)
region_central_hex_total = hex_pipe_prism & +z_bot_core & -z_top_core

# 2. Definiamo la regione del TARGET SOLIDO (Cilindro limitato in Z)
# Nota: Intersechiamo con region_central_hex_total per sicurezza, ma basterebbe il cilindro.
region_target_solid = region_central_hex_total & -cyl_target & +z_tgt_bot & -z_tgt_top

# 3. Definiamo la regione del VUOTO (Tutto il canale MENO il target)
# Usiamo l'operatore ~ (NOT) sulla regione del target.
region_void_gap = region_central_hex_total & ~region_target_solid

# Creazione delle celle
c_target = openmc.Cell(fill=mat_target, region=region_target_solid)
c_void_gap = openmc.Cell(fill=void_air, region=region_void_gap) 

# C. Riflettori (Rimangono uguali)
region_ref_side = prism_reflector & ~prism_core & +z_bot_core & -z_top_core
c_ref_side = openmc.Cell(fill=reflector, region=region_ref_side)

region_ref_bot = prism_reflector & +z_bot_ref & -z_bot_core
c_ref_bot = openmc.Cell(fill=reflector, region=region_ref_bot)

# D. Top World & Source (Rimangono uguali)
c_source_marker = openmc.Cell(fill=marker_mat, region=-sphere_src)
region_top = prism_reflector & +z_top_core & -z_top_world & +sphere_src
c_top_void = openmc.Cell(fill=void_air, region=region_top)

all_cells = [c_main_core, c_target, c_void_gap, c_ref_side, c_ref_bot, c_source_marker, c_top_void]
geometry = openmc.Geometry(all_cells)
geometry.export_to_xml()
print("Geometria generata correttamente.")

def count_fuel_pins(num_rings, pitch, r_pipe, r_core):
    total_pins = 0
    # Ring 0 è il centro
    for i in range(num_rings):
        ring_radius = i * pitch
        # Se l'anello è fuori dalla pipe centrale e dentro il raggio del core
        if ring_radius >= r_pipe and ring_radius <= r_core:
            if i == 0:
                total_pins += 1
            else:
                total_pins += (6 * i)
    return total_pins

n_barre = count_fuel_pins(num_rings, PITCH, R_PIPE, R_CORE)
print(f"Numero di barre di combustibile effettive: {n_barre} \n\n")

Geometria generata correttamente.
Numero di barre di combustibile effettive: 390 




/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/model/funcs.py:130: FutureWarning: get_hexagonal_prism(...) has been renamed hexagonal_prism(...). Future versions of OpenMC will not accept get_hexagonal_prism.
  warn("get_hexagonal_prism(...) has been renamed hexagonal_prism(...). "
/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/model/funcs.py:121: FutureWarning: The hexagonal_prism(...) function has been replaced by the HexagonalPrism(...) class. Future versions of OpenMC will not accept hexagonal_prism.
  warn("The hexagonal_prism(...) function has been replaced by the "


PLOTTING


SETTING EIGENVALUED

In [47]:
settings_k = openmc.Settings()
settings_k.run_mode = 'eigenvalue'
settings_k.batches = batches
settings_k.particles = particles
settings_k.inactive = inactive

# Definiamo un box che racchiude l'intero core
lower_left = [-R_CORE, -R_CORE, 0.0]
upper_right = [R_CORE, R_CORE, H_CORE]

# Creazione di una distribuzione spaziale uniforme confinata ai materiali fessili
uniform_dist = openmc.stats.Box(lower_left, upper_right, only_fissionable=True)

settings_k.source = openmc.IndependentSource(space=uniform_dist)

# Override della frazione di rigetto
settings_k.source_rejection_fraction = 0.001 
settings_k.temperature = {'method': 'interpolation'}

# Esportazione
settings_k.export_to_xml()


/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/stats/multivariate.py:1115: FutureWarning: The 'only_fissionable' has been deprecated. Use the 'constraints' argument when defining a source instead.
  warn("The 'only_fissionable' has been deprecated. Use the "


In [48]:
# --- INIZIALIZZAZIONE LISTA TALLIES ---
tallies = openmc.Tallies()

# --- 1. FILTRI ENERGETICI PER U-238 ---
# Gruppi: Termico, Epitermico (Risonanze U-238), Veloce
e_groups = openmc.EnergyFilter([0.0, 0.625, 100.0e3, 20.0e6])

# Filtro sul materiale combustibile (nel notebook si chiama 'fuel')
fuel_mat_filter = openmc.MaterialFilter(fuel)

tally_rr_groups = openmc.Tally(name='fuel_rr_groups') 
tally_rr_groups.filters = [fuel_mat_filter, e_groups] 
tally_rr_groups.scores = [ 'fission', # Tasso di fissione (fission rate) 
                          'nu-fission', # Tasso di produzione neutroni da fissione 
                          '(n,gamma)', # Cattura radiativa (fondamentale su U-238) 
                          'absorption', # Assorbimento totale 
                          'flux', # Flusso neutronico nel combustibile 
                          ] 
tallies.append(tally_rr_groups) 

# --- TALLY 2: SPETTRO CONTINUO DEL TASSO DI FISSIONE E CATTURA (300 bin) --- 

energies_fine = np.logspace(-5, np.log10(20.0e6), 301) 
tally_rr_spec = openmc.Tally(name='fuel_rr_spectrum') 
tally_rr_spec.filters = [fuel_mat_filter, openmc.EnergyFilter(energies_fine)] 
tally_rr_spec.scores = ['fission', '(n,gamma)'] 
tallies.append(tally_rr_spec)

tallies.export_to_xml()

In [49]:
from CoolProp.CoolProp import PropsSI

def densita_mix_dinamica(T_K, P_Pa=pressione_funzionamento, frac_mass_H2O=mix):
    # Calcolo in kg/m^3 e conversione in g/cm^3
    rho_H2O = PropsSI('D', 'T', T_K, 'P', P_Pa, 'Water') / 1000     # D sta per densità
    rho_D2O = PropsSI('D', 'T', T_K, 'P', P_Pa, 'HeavyWater') / 1000
    
    # Calcolo miscela assumendo volumi ideali additivi
    rho_mix = 1 / (frac_mass_H2O / rho_H2O + (1 - frac_mass_H2O) / rho_D2O) # Formula di additività dei volumi specifici
    return rho_mix

In [50]:
now = datetime.now()
output_file = "result.txt"

# Scrittura Header condizionata
if is_first_run:
    timestamp = now.strftime("\n\n------ SIMULAZIONE DEL %d/%m/%Y ALLE ORE %H:%M ------\n")
    with open(output_file, "a") as f_out:
        f_out.write(timestamp)
        f_out.write("=== PARAMETRI DI SIMULAZIONE ===\n")
        f_out.write(f"intensità sorgente: {S} fotoni/s\n")
        f_out.write(f"Arricchimento: {arricchimento}\n")
        f_out.write(f"percentuale di acqua leggera: {mix*100:.2f} % \n")
        f_out.write(f"Target (R, H, Pos): {R_TARGET}, {H_TARGET}, {P_TARGET} cm\n")
        f_out.write(f"Settings: {batches} batches, {particles} particles, {inactive} inactive batches\n")
        f_out.write("================================\n\n")
        f_out.write(f"{'Pres [Pa]':<12} {'PITCH [cm]':<12} {'T_WATER [K]':<12} {'T_FUEL [K]':<12} {'K_EFF':<12} {'STD_DEV':<12}\n")

temperature_pairs = [
    (300, 300),                 
    (300, 350),
    (300, 400),
    (300, 450),
    (300, 500),
    (300, 550),
    (300, 600),
]

for T_water, T_fuel in temperature_pairs:
    
    # Aggiornamento proprietà moderatore
    water.temperature = T_water
    water.set_density('g/cm3', densita_mix_dinamica(T_water))
    
    # Aggiornamento temperature elementi solidi
    print(f"Simulazione: H2O={T_water}K, Fuel={T_fuel}K")
    fuel.temperature = T_fuel
    cladding.temperature = T_fuel
    reflector.temperature = T_fuel
    mat_target.temperature = T_fuel

    # Esportazione materiali
    mats = openmc.Materials([fuel, cladding, water, reflector, marker_mat, void_air, mat_target])
    materials.cross_sections = "/home/bossi_ricky/openmc_data/mcnp_endfb71/cross_sections.xml"
    mats.export_to_xml()
    
    # Esecuzione OpenMC
    openmc.run( output=False)
    sp_filename = f'statepoint.{batches}.h5'

    # Estrazione risultati e salvataggio
    if os.path.exists(sp_filename):
        with openmc.StatePoint(sp_filename) as sp:
            k_val = sp.k_combined.nominal_value
            k_std = sp.k_combined.std_dev

            print(f"valore di k: {k_val:.5f} +/- {k_std:.5f}")

            with open(output_file, "a") as f_out:
                f_out.write(f"{pressione_funzionamento:<12}{PITCH:<12.3f}{T_water:<12} {T_fuel:<12} {k_val:<12.5f} {k_std:<12.5f}\n")
        
        # INVECE DI ELIMINARE IL FILE, LO RINOMINIAMO IN MODO UNIVOCO
        # Così `analisi.ipynb` potrà caricare tutti i file generati e fare i plot.
        nuovo_nome_sp = f"statepoint_M{moltiplicatore}_P{pressione_funzionamento}_W{mix}_T{T_fuel}.h5"
        os.rename(sp_filename, nuovo_nome_sp)

    if os.path.exists('summary.h5'):
        os.remove('summary.h5')

Simulazione: H2O=300K, Fuel=300K


/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/statepoint.py:289: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(


valore di k: 0.98058 +/- 0.00031
Simulazione: H2O=300K, Fuel=350K


/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/statepoint.py:289: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(


valore di k: 0.97982 +/- 0.00030
Simulazione: H2O=300K, Fuel=400K


...

/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/statepoint.py:289: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(


valore di k: 0.97747 +/- 0.00030
Simulazione: H2O=300K, Fuel=550K


/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/statepoint.py:289: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(


valore di k: 0.97650 +/- 0.00032
Simulazione: H2O=300K, Fuel=600K
valore di k: 0.97610 +/- 0.00033


/home/laptop_boz/miniconda3/envs/openmc-env/lib/python3.12/site-packages/openmc/statepoint.py:289: FutureWarning: The 'k_combined' property has been renamed to 'keff' and will be removed in a future version of OpenMC.
  warnings.warn(
